In [4]:
# ==============================================================================
# FULL PIPELINE: MACAQUE & HUMAN REPRESENTATIONAL GEOMETRY (RSA)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. IMPORTS & SETUP
# ------------------------------------------------------------------------------
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import seaborn as sns
from pynwb import NWBHDF5IO
from tqdm import tqdm
import mne  # Requires mne to be installed: pip install mne

warnings.filterwarnings('ignore')

# Macaque Setup (Watters TensorFlow Dataset)
sys.path.append('../multi_object_memory_2025')
sys.path.append('../multi_object_memory_2025/phys_modeling')
sys.path.append('../multi_object_memory_2025/phys_modeling/training')
try:
    from phys_modeling.training.dataset import Dataset
except ImportError:
    print("Warning: Could not import Dataset. Ensure Macaque dataset paths are correct.")

# ------------------------------------------------------------------------------
# 2. DATA EXTRACTION FUNCTIONS
# ------------------------------------------------------------------------------
def get_sliding_window_counts(spike_times, delay_start, delay_end, window_size=0.5, step_size=0.05):
    spike_times = np.array(spike_times)
    bin_starts = np.arange(delay_start, delay_end - window_size, step_size)
    binned_counts = np.zeros(len(bin_starts))
    bin_centers = bin_starts + (window_size / 2.0)
    for i, t_start in enumerate(bin_starts):
        t_end = t_start + window_size
        binned_counts[i] = np.sum((spike_times >= t_start) & (spike_times < t_end))
    return binned_counts, bin_centers

def extract_time_resolved_human_data(human_data_root, custom_cache_dir, target_regions, window_size=0.5, step_size=0.05):
    human_cells_dict = {}
    master_time_vector = None 
    
    csv_files = glob.glob(os.path.join(custom_cache_dir, "*_concept_cells.csv"))
    print(f"Found {len(csv_files)} concept cell CSVs for Human Extraction.")
    
    for csv_path in tqdm(csv_files, desc="Processing Human CSVs"):
        df_concepts = pd.read_csv(csv_path)
        df_target = df_concepts[df_concepts['location'].isin(target_regions)]
        if df_target.empty: continue
            
        parts = os.path.basename(csv_path).split('_')
        sub_id, ses_id = parts[0], parts[1]
        
        nwb_matches = glob.glob(os.path.join(human_data_root, "**", f"{sub_id}_{ses_id}*WorkingMemory*.nwb"), recursive=True)
        if not nwb_matches: nwb_matches = glob.glob(os.path.join(human_data_root, "**", f"{sub_id}_{ses_id}*.nwb"), recursive=True)
        if not nwb_matches: continue
            
        with NWBHDF5IO(nwb_matches[0], 'r') as io:
            nwb = io.read()
            trials_df = nwb.trials.to_dataframe().dropna(subset=['timestamps_Maintenance', 'timestamps_Probe'])
            
            for _, cell_row in df_target.iterrows():
                if pd.isna(cell_row['preferred_image_id']): continue
                    
                unit_id = int(cell_row['unit_id'])
                location = cell_row['location']
                pref_id = int(cell_row['preferred_image_id'])
                
                try: spike_times = nwb.units.get_unit_spike_times(unit_id)
                except: continue
                
                X_trials, y_binned_list, loads_list = [], [], []
                
                for _, trial in trials_df.iterrows():
                    d_start = trial['timestamps_Maintenance']
                    d_end = trial['timestamps_Probe']
                    load = int(trial['loads'])
                    
                    counts, centers = get_sliding_window_counts(spike_times, d_start, d_end, window_size, step_size)
                    if master_time_vector is None: master_time_vector = centers - d_start 
                    
                    target_len = len(master_time_vector)
                    if len(counts) > target_len: counts = counts[:target_len]
                    elif len(counts) < target_len: counts = np.pad(counts, (0, target_len - len(counts)), 'constant')
                        
                    seq = [0, 0, 0] 
                    if load >= 1: seq[0] = int(trial['loadsEnc1_PicIDs'])
                    if load >= 2: seq[1] = int(trial['loadsEnc2_PicIDs'])
                    if load >= 3: seq[2] = int(trial['loadsEnc3_PicIDs'])
                    
                    X_trials.append(seq)
                    y_binned_list.append(counts)
                    loads_list.append(load)
                    
                cell_key = f"{sub_id}_{ses_id}_unit{unit_id}_{location}"
                human_cells_dict[cell_key] = (np.array(X_trials), np.array(y_binned_list), np.array(loads_list), pref_id)

    print(f"Human Extraction complete! Formatted {len(human_cells_dict)} concept cells.")
    return human_cells_dict, master_time_vector

def extract_time_resolved_macaque_data(session_list):
    macaque_cells_dict = {}
    concept_map = {'a': 1, 'b': 2, 'c': 3}
    master_time_vector = None
    
    # Identity-Only Filter
    identity_only_filter = lambda df: df[
        (df['quality'].isin(['good', '1', 'sua'])) & 
        (df['Identity AUC'] > 0.65) & 
        (df['Identity p-value'] < 0.05) & 
        (df['probe'] == 's0')
    ]
    
    print(f"Extracting Macaque Data for {len(session_list)} sessions...")
    for subject, session in tqdm(session_list, desc="Processing Macaque Sessions"):
        try:
            ds_delay = Dataset(subject=subject, session=session, phase="delay", 
                               unit_filter=identity_only_filter, trial_filter=lambda df: df, max_n_objects=3)
                               
            raw_neural = ds_delay.data['neural'] 
            if raw_neural.shape[2] == 0: continue
                
            trials_df = None
            if isinstance(ds_delay.data, dict):
                for key, val in ds_delay.data.items():
                    if isinstance(val, pd.DataFrame): trials_df = val; break       
            if trials_df is None:
                for attr in dir(ds_delay):
                    val = getattr(ds_delay, attr)
                    if isinstance(val, pd.DataFrame): trials_df = val; break
                        
            if trials_df is None: continue
                
            if len(trials_df) != raw_neural.shape[0]:
                if 'completed' in trials_df.columns:
                    trials_df = trials_df[trials_df['completed'] == True]
            
            # THE FIX: Ensure the time vector we grab actually matches the bin count
            num_bins = raw_neural.shape[1]
            if master_time_vector is None:
                if 'time' in ds_delay.data and len(ds_delay.data['time']) == num_bins: 
                    master_time_vector = ds_delay.data['time']
                elif 'time_delay_onset' in trials_df.columns and 'time_cue_onset' in trials_df.columns:
                    median_delay = (trials_df['time_cue_onset'] - trials_df['time_delay_onset']).median()
                    master_time_vector = np.linspace(0, median_delay, num_bins)
                else:
                    master_time_vector = np.linspace(0, 1.0, num_bins)
            
            X_trials, loads_list = [], []
            for _, row in trials_df.iterrows():
                spatial_array = [0, 0, 0]
                load = int(row['num_objects'])
                if load >= 1 and pd.notna(row['object_0_location']): spatial_array[int(row['object_0_location'])] = concept_map[row['object_0_id']]
                if load >= 2 and pd.notna(row['object_1_location']): spatial_array[int(row['object_1_location'])] = concept_map[row['object_1_id']]
                if load >= 3 and pd.notna(row['object_2_location']): spatial_array[int(row['object_2_location'])] = concept_map[row['object_2_id']]
                    
                X_trials.append(spatial_array)
                loads_list.append(load)
                
            X_trials, loads_list = np.array(X_trials), np.array(loads_list)
            n_neurons = raw_neural.shape[2]
            
            for neuron_idx in range(n_neurons):
                y_binned = raw_neural[:, :, neuron_idx]
                macaque_cells_dict[f"{subject}_{session}_unit{neuron_idx}"] = (X_trials, y_binned, loads_list)
                
        except Exception:
            pass 
            
    print(f"Macaque Extraction complete! Formatted {len(macaque_cells_dict)} Identity-tuned DMFC cells.")
    
    if master_time_vector is None and len(macaque_cells_dict) > 0:
        sample_key = list(macaque_cells_dict.keys())[0]
        num_bins = macaque_cells_dict[sample_key][1].shape[1]
        master_time_vector = np.linspace(0, 1.0, num_bins) 
        
    return macaque_cells_dict, master_time_vector



# ------------------------------------------------------------------------------
# 3. RSA & THEORETICAL MODEL FUNCTIONS
# ------------------------------------------------------------------------------
def create_macaque_euclidean_models():
    """Builds the 27x27 Theoretical RDMs using generative feature vectors."""
    conds = [(l, p, i) for l in [1,2,3] for p in [0,1,2] for i in [1,2,3]]
    n = len(conds)
    
    f_identity = np.zeros((n, 3)) # 3 dimensions (one per item)
    f_slot = np.zeros((n, 9))     # 9 dimensions (3 items x 3 positions)
    f_gain = np.zeros((n, 3))     # 3 dimensions (magnitude scales with load)
    
    for idx, (load, pos, item) in enumerate(conds):
        i_idx = item - 1
        p_idx = pos
        slot_idx = (p_idx * 3) + i_idx
        
        f_identity[idx, i_idx] = 1.0
        f_slot[idx, slot_idx] = 1.0
        f_gain[idx, i_idx] = 1.0 / load 

    rdm_identity = squareform(pdist(f_identity, metric='euclidean'))
    rdm_slot = squareform(pdist(f_slot, metric='euclidean'))
    rdm_gain = squareform(pdist(f_gain, metric='euclidean'))
    
    return rdm_identity, rdm_slot, rdm_gain

def compute_bootstrapped_euclidean_rsa(cells_dict, time_bins, conditions_list, n_bootstraps=100):
    """
    Computes Euclidean distance RDMs with cell-level bootstrapping.
    Handles both Human (Pref item) and Macaque (Explicit positions) data.
    """
    cell_keys = list(cells_dict.keys())
    num_cells = len(cell_keys)
    n_conds = len(conditions_list)
    
    # THE FIX: Derive the true number of time bins directly from the empirical data
    sample_data = cells_dict[cell_keys[0]]
    true_num_bins = sample_data[1].shape[1]
    num_bins = true_num_bins
    
    # Safely align time_bins to empirical data length if it somehow mismatch
    if time_bins is not None:
        if len(time_bins) > num_bins:
            time_bins = np.array(time_bins)[:num_bins]
        elif len(time_bins) < num_bins:
            time_bins = np.pad(time_bins, (0, num_bins - len(time_bins)), 'linear_ramp')
    
    master_tensor = np.full((n_conds, num_bins, num_cells), np.nan)
    
    print(f"Aggregating PSTHs for {num_cells} cells across {n_conds} conditions...")
    for n_idx, cell_key in enumerate(cell_keys):
        data = cells_dict[cell_key]
        is_human = len(data) == 4
        
        if is_human:
            X, y_binned, loads, pref_id = data
            is_present = np.any(X == pref_id, axis=1)
        else:
            X, y_binned, loads = data
            
        for row_idx, cond in enumerate(conditions_list):
            if is_human:
                parts = cond.split('_')
                L = int(parts[0][1])
                mask = (loads == L) & (is_present if parts[1] == 'Pres' else ~is_present)
            else:
                parts = cond.split('_')
                L, P, I = int(parts[0][1]), int(parts[1][1]), int(parts[2][1])
                mask = (loads == L) & (X[:, P] == I)
                
            if np.sum(mask) > 0:
                master_tensor[row_idx, :, n_idx] = np.nanmean(y_binned[mask, :], axis=0)

    bootstrapped_rdms = np.zeros((n_bootstraps, num_bins, n_conds, n_conds))
    
    print(f"Bootstrapping Euclidean RDMs ({n_bootstraps} iterations)...")
    for b in tqdm(range(n_bootstraps), leave=False):
        boot_idx = np.random.choice(num_cells, num_cells, replace=True)
        boot_tensor = master_tensor[:, :, boot_idx]
        
        for t in range(num_bins):
            pop_matrix = boot_tensor[:, t, :]
            
            # Nan-Euclidean distance to handle unmeasured conditions cleanly
            diffs = pop_matrix[:, None, :] - pop_matrix[None, :, :]
            sq_diffs = np.square(diffs)
            mean_sq_dist = np.nanmean(sq_diffs, axis=2) 
            euclidean_dist = np.sqrt(mean_sq_dist * num_cells)
            
            bootstrapped_rdms[b, t, :, :] = euclidean_dist
            
    mean_rdms = np.nanmean(bootstrapped_rdms, axis=0)
    
    # Optional update: return the safely trimmed time_bins
    return mean_rdms, bootstrapped_rdms, time_bins


def get_macaque_identity_dist(bootstrapped_rdms, load, idx_map):
    """ Extracts within-load, cross-item distances for Macaque. """
    dists = []
    for p in [0, 1, 2]:
        idx_1 = idx_map[f"L{load}_P{p}_I1"]
        idx_2 = idx_map[f"L{load}_P{p}_I2"]
        idx_3 = idx_map[f"L{load}_P{p}_I3"]
        dists.extend([
            bootstrapped_rdms[:, :, idx_1, idx_2],
            bootstrapped_rdms[:, :, idx_1, idx_3],
            bootstrapped_rdms[:, :, idx_2, idx_3]
        ])
    return np.nanmean(dists, axis=0) # Shape: (n_boots, n_time)

# ------------------------------------------------------------------------------
# 4. STATISTICAL FUNCTIONS (MNE CLUSTER PERMUTATION)
# ------------------------------------------------------------------------------
def run_cluster_stats(bootstrapped_ts_A, bootstrapped_ts_B):
    """
    Runs a proper 1D cluster-based permutation test using MNE.
    Inputs are shape: (n_bootstraps, n_timebins)
    """
    diff = bootstrapped_ts_A - bootstrapped_ts_B
    
    # MNE tests difference against 0 via random sign flips
    t_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_1samp_test(
        diff, n_permutations=1000, tail=0, out_type='mask', verbose=False
    )
    
    sig_bins = []
    if clusters:
        for i, c in enumerate(clusters):
            if cluster_pv[i] < 0.05:
                sig_bins.extend(np.where(c)[0].tolist())
                
    return np.unique(sig_bins)

# ------------------------------------------------------------------------------
# 5. EXECUTION & PLOTTING PIPELINE
# ------------------------------------------------------------------------------

# --- A. Data Extraction ---
human_target_regions = ['pre_supplementary_motor_area_right', 'pre_supplementary_motor_area_left', 
                        'dorsal_anterior_cingulate_cortex_right', 'dorsal_anterior_cingulate_cortex_left']
human_data_root = "/Volumes/fetty/000469"
custom_cache_dir = '/Users/cwook/Documents/humac/data/concept_cells'

print("\n--- Starting Human Extraction ---")
tr_human_dict, human_time_bins = extract_time_resolved_human_data(human_data_root, custom_cache_dir, human_target_regions)

print("\n--- Starting Macaque Extraction ---")
try:
    mac_shape = pd.read_csv('/Users/cwook/Documents/humac/lists/task_shape_inventory.csv')
    macaque_sessions = [(r['file'].split('/')[0].replace('sub-',''), r['file'].split('/')[1].split('_')[1].replace('ses-','')) 
                        for n,r in mac_shape[mac_shape['shape'] == 'Triangle'].iterrows()]
    tr_mac_dict, mac_time_bins = extract_time_resolved_macaque_data(macaque_sessions)
except FileNotFoundError:
    print("Warning: task_shape_inventory.csv not found. Skipping Macaque extraction.")
    tr_mac_dict, mac_time_bins = {}, []

# --- B. RSA Execution ---
human_conds = ["L1_Pres", "L2_Pres", "L3_Pres", "L1_Abs", "L2_Abs", "L3_Abs"]
print("\n--- Running Bootstrapped RSA (Human) ---")
if tr_human_dict:
    hum_mean_rdms, hum_boot_rdms, human_time_bins = compute_bootstrapped_euclidean_rsa(tr_human_dict, human_time_bins, human_conds, n_bootstraps=500)    
    # Extract Human PvA Contrasts
    dist_L1_hum_boot = hum_boot_rdms[:, :, 0, 3] # L1_Pres vs L1_Abs
    dist_L2_hum_boot = hum_boot_rdms[:, :, 1, 4] # L2_Pres vs L2_Abs
    dist_L3_hum_boot = hum_boot_rdms[:, :, 2, 5] # L3_Pres vs L3_Abs

mac_conds = sorted([f"L{l}_P{p}_I{i}" for l in [1,2,3] for p in [0,1,2] for i in [1,2,3]])
mac_idx_map = {cond: i for i, cond in enumerate(mac_conds)}

print("\n--- Running Bootstrapped RSA (Macaque) ---")
if tr_mac_dict:
    mac_mean_rdms, mac_boot_rdms, mac_time_bins = compute_bootstrapped_euclidean_rsa(tr_mac_dict, mac_time_bins, mac_conds, n_bootstraps=500)    
    # Extract Macaque Identity Distances
    dist_L1_mac_boot = get_macaque_identity_dist(mac_boot_rdms, 1, mac_idx_map)
    dist_L2_mac_boot = get_macaque_identity_dist(mac_boot_rdms, 2, mac_idx_map)
    dist_L3_mac_boot = get_macaque_identity_dist(mac_boot_rdms, 3, mac_idx_map)

# --- C. Stats & Plotting ---
sns.set_theme(style="whitegrid", context="talk")

if tr_human_dict and len(human_time_bins) > 0:
    sig_human_bins = run_cluster_stats(dist_L1_hum_boot, dist_L3_hum_boot)
    
    plt.figure(figsize=(10, 6))
    
    mean_L1 = np.nanmean(dist_L1_hum_boot, axis=0)
    ci_L1 = np.nanstd(dist_L1_hum_boot, axis=0) * 1.96
    plt.plot(human_time_bins, mean_L1, label="Load 1", color='#2ca02c')
    plt.fill_between(human_time_bins, mean_L1 - ci_L1, mean_L1 + ci_L1, color='#2ca02c', alpha=0.2)

    mean_L3 = np.nanmean(dist_L3_hum_boot, axis=0)
    ci_L3 = np.nanstd(dist_L3_hum_boot, axis=0) * 1.96
    plt.plot(human_time_bins, mean_L3, label="Load 3", color='#d62728')
    plt.fill_between(human_time_bins, mean_L3 - ci_L3, mean_L3 + ci_L3, color='#d62728', alpha=0.2)
    
    if len(sig_human_bins) > 0:
        plt.scatter(np.array(human_time_bins)[sig_human_bins], 
                    np.zeros_like(sig_human_bins) + np.nanmin(mean_L3)*0.9, 
                    marker='s', color='black', label='p < 0.05 (L1 > L3)')
                    
    plt.title("Human SMA/dACC: Present vs Absent Distance (Bootstrapped Euclidean)")
    plt.xlabel("Time from Delay Onset (s)")
    plt.ylabel("Euclidean Distance")
    plt.legend(loc='upper right')
    sns.despine()
    plt.savefig("Figure_4_Human_Euclidean_RSA.pdf", format='pdf', bbox_inches='tight')
    plt.show()

if tr_mac_dict and len(mac_time_bins) > 0:
    sig_mac_bins = run_cluster_stats(dist_L1_mac_boot, dist_L3_mac_boot)
    
    plt.figure(figsize=(10, 6))
    
    m_mean_L1 = np.nanmean(dist_L1_mac_boot, axis=0)
    m_ci_L1 = np.nanstd(dist_L1_mac_boot, axis=0) * 1.96
    plt.plot(mac_time_bins, m_mean_L1, label="Load 1", color='#2ca02c')
    plt.fill_between(mac_time_bins, m_mean_L1 - m_ci_L1, m_mean_L1 + m_ci_L1, color='#2ca02c', alpha=0.2)

    m_mean_L3 = np.nanmean(dist_L3_mac_boot, axis=0)
    m_ci_L3 = np.nanstd(dist_L3_mac_boot, axis=0) * 1.96
    plt.plot(mac_time_bins, m_mean_L3, label="Load 3", color='#d62728')
    plt.fill_between(mac_time_bins, m_mean_L3 - m_ci_L3, m_mean_L3 + m_ci_L3, color='#d62728', alpha=0.2)
    
    if len(sig_mac_bins) > 0:
        plt.scatter(np.array(mac_time_bins)[sig_mac_bins], 
                    np.zeros_like(sig_mac_bins) + np.nanmin(m_mean_L3)*0.9, 
                    marker='s', color='black', label='p < 0.05 (L1 > L3)')
                    
    plt.title("Macaque DMFC: Identity Distance (Bootstrapped Euclidean)")
    plt.xlabel("Time from Delay Onset (s)")
    plt.ylabel("Euclidean Distance")
    plt.legend(loc='upper right')
    sns.despine()
    plt.savefig("Figure_3_Macaque_Euclidean_RSA.pdf", format='pdf', bbox_inches='tight')
    plt.show()

print("\nPipeline execution finished successfully.")


--- Starting Human Extraction ---
Found 17 concept cell CSVs for Human Extraction.


Processing Human CSVs: 100%|██████████| 17/17 [00:04<00:00,  3.47it/s]


Human Extraction complete! Formatted 17 concept cells.

--- Starting Macaque Extraction ---
Extracting Macaque Data for 24 sessions...


Processing Macaque Sessions:   0%|          | 0/24 [00:00<?, ?it/s]

Triangle session
Original number of trials: 2267
Original number of units: 203


Processing Macaque Sessions:   4%|▍         | 1/24 [00:08<03:08,  8.21s/it]

Number of trials after filtering: 0
Number of units after filtering: 0
Triangle session
Original number of trials: 1512
Original number of units: 151
